# LoRA Dialect Fine-Tuner

Fine-tune a language model to adopt a style — training only a tiny LoRA adapter on top of a frozen base. **Runtime: GPU.**

## 1. Install libraries

In [ ]:
!pip -q install transformers peft datasets accelerate

## 2. Load a small base model
We use distilgpt2 so it runs fast on a free GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = 'distilgpt2'
tok = AutoTokenizer.from_pretrained(name)
tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(name)
print('base params:', sum(p.numel() for p in base.parameters()))

## 3. Wrap it with LoRA
PEFT freezes the base and injects small trainable low-rank matrices.

In [ ]:
from peft import LoraConfig, get_peft_model
cfg = LoraConfig(r=8, lora_alpha=16, target_modules=['c_attn'], lora_dropout=0.05, task_type='CAUSAL_LM')
model = get_peft_model(base, cfg)
model.print_trainable_parameters()  # note how tiny the trainable % is

## 4. A tiny instruction dataset
Replace these with examples in your target style/dialect.

In [ ]:
samples = [
    'Q: How are you? A: Arr, I be doin grand, matey!',
    'Q: What is the weather? A: The skies be clear, ye lucky sailor!',
    'Q: Tell me a fact. A: Arr, the ocean covers most o the globe!',
]
enc = tok(samples, return_tensors='pt', padding=True)
enc['labels'] = enc['input_ids'].clone()

## 5. Train the adapter

In [ ]:
model.train()
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
for step in range(60):
    out = model(**enc)
    out.loss.backward(); opt.step(); opt.zero_grad()
    if step % 20 == 0:
        print(step, round(out.loss.item(), 3))

## 6. Generate in the new style

In [ ]:
model.eval()
ids = tok('Q: How are you? A:', return_tensors='pt')
print(tok.decode(model.generate(**ids, max_new_tokens=20)[0]))

## Homework
1. Build a bigger, cleaner style dataset and retrain.
2. Save just the adapter (`model.save_pretrained`) — note how small it is (a few MB).
3. Load a different base model and reuse the same adapter idea.
4. Compare outputs before vs. after tuning, side by side.